# Observability& Evaluation

**Module 14 · Lesson 14.3**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Traces vs Logs
- The Golden Set
- Code-Based Asserts
- LLM-As-Judge
- Judge Calibration
- The CI Eval Gate

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install anthropic -q

# Reproducibility - the lesson uses seeded random values throughout

## “Did the model do a good job?” — the hardest question in LLMOps

> 💡 **Info**
>
> On classical machine learning you score a prediction against a label and compute an F1. On LLMs the output is **prose**, and “good” depends on the customer, the context, and what the user actually wanted — so there is no single number to compute. That is why every earlier lesson in this module ended by pointing here: 14.1 gave you the trace record, 14.2 gave you the A/B test, and both stopped at the same wall — *which arm actually won?* This lesson answers it by breaking the impossible question into three you *can* answer. **What did the model actually do?** is **tracing**: one request as a tree of timed, priced sub-steps. **Did it do the right thing on a curated set?** is the **golden set** — versioned examples with known answers — screened by cheap **code asserts** and graded on the subjective parts by an **LLM-as-judge** that you **calibrate** against humans. And **is it still doing the right thing today?** is a **CI eval gate** on every change plus **drift detection** on live traffic. Master those three and you are the level-3 LLMOps engineer the maturity model from 14.1 described.

### 🎯 What you will be able to do after this lesson

- **Build** the eval toolkit as runnable models — a span-tree trace, a golden-set runner, a code-assert screen, an LLM-as-judge rubric, a Cohen’s-kappa calibration check, a three-part CI eval gate, and a nightly drift monitor.

- **Compare** a flat log with a trace, a free code assert with a paid judge, and raw agreement with chance-corrected kappa.

- **Analyze** an eval result for calibration and bias *before* you trust the judge’s scores.

- **Evaluate** whether a prompt change is safe to merge, and whether quality has silently drifted with nothing in your code changing.

> 📦 **Info**
>
> ✅ Before you startThis is the payoff lesson. In **14.1** every call emitted a flat trace record; here it becomes a tree of spans. In **14.2** the A/B test ended on a “pass” a judge had to define; here you build and calibrate that judge. In **8.11** RAGAS gave you automated eval metrics for retrieval; the LLM-as-judge is the general version. And the significance math that decides whether a lift is real is Lesson 14.2 — here you use its conclusion (the gate thresholds), not re-derive it. Deciding *what* the judge should even measure, from a taxonomy of real failures, is the next lesson.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> 🧾 **Analogy**
>
> “How good was this?” is impossible to answer in one number, so you do what a good clinic does with “how healthy is this patient?”: you **break it into measurable pieces**. You do not eyeball the patient and write down “7 out of 10”. You take the vitals (that is tracing — the timed, priced sub-steps), you run the standard panel of tests against known reference ranges (that is the golden set with its answer key), you have a specialist read the ambiguous scans (that is the LLM-as-judge), and you **re-check the specialist against a senior consultant** before you trust their reads (that is calibration). **Where it breaks down:** a lab test has an objective ground truth; an LLM judge is itself a fallible model, which is exactly why you never skip the calibration step.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“A high pass rate means it is good.”** A golden set of only easy cases lets a mediocre prompt score perfectly while it quietly fails every hard, ambiguous, or adversarial case. The headline number hides the weakness.
> - **“The judge agrees with us most of the time, so trust it.”** Raw agreement flatters because two labellers agree by chance a lot. A judge at eighty-three percent agreement can have a Cohen’s kappa around 0.56 — borderline. Correct for chance before you trust the scores.

> 💡 **Info**
>
> ⚠️ Anti-patternThe judge that feels rigorous and is worthless: it grades on a free *one-to-ten* scale (so every output lands at seven), it uses the *same model family* as the thing it is grading (so it over-rewards its own style), it has *no reference answer* in the prompt (so it guesses what “correct” means), and it was *never checked against humans* (so you have confident numbers and no idea if they are right). Every step below exists to prevent one of those four.

---

## 🎯 Concept 1: Traces vs Logs

### Traces vs Logs

A modern LLM call is rarely one API request - it is embed, retrieve, build-messages, model call, parse, log. A flat log gives one line (the total); a TRACE breaks the request into per-step SPANS with their own timing and cost. Summing the spans gives the same total, but only the trace tells you WHERE the time went - so you optimise the right thing.

Start with seeing what happened. A classical web request gets by with one log line, but an LLM call is a small pipeline: load the prompt, embed the input, retrieve context, build the messages, call the model, parse the output, log the outcome. If your monitoring is a single line that says *“latency: one-point-two seconds”*, you cannot tell whether the slow part was retrieval, the model, or the database write — and that is the difference between debugging in five minutes and five hours. A **trace** is the full record of one request; a **span** is one sub-operation inside it; an **attribute** is metadata on a span (model, tokens, prompt version). Spans link parent to child to form a tree, and each carries its own timing and cost. Crucially, summing the spans gives the same total a flat log would — but only the trace localises it. The block builds the span tree and finds the real bottleneck, keyless. (The field set on each span is the OpenTelemetry GenAI convention you met in 14.1; in production you use a tool’s wrapper rather than hand-rolling it.)

> 🧾 **Analogy**
>
> A trace is **an itemized receipt instead of just the total at the bottom**. When you get a bill that says only “forty-seven dollars”, you cannot tell whether the expensive thing was the entree, the wine, or a service charge — you just know the total. An itemized receipt lists every line with its own price, so you can see instantly that the wine was two-thirds of the bill. A flat log is the total; a trace is the itemization. And just like a receipt, the line items add up to exactly the same total — the point is not a different number, it is being able to see *which line* is the big one.

A trace shows your one-and-a-quarter-second request as spans: retrieval took a few tens of milliseconds, the model call took over a second. Where do you optimise?

**📝 `01_traces_vs_logs.py`** - *runnable*

In [ ]:
# A trace is the whole request broken into SPANS (sub-operations); a flat log is one line. Summing the spans gives
# the same total a log would, but ONLY the trace tells you WHERE the time went - so you optimise the right thing.
spans = [("load_prompt", 3), ("embed_review", 48), ("retrieve_similar", 31), ("build_messages", 2),
         ("anthropic.messages.create", 1140), ("parse_json", 1), ("log_outcome", 22)]   # ms per span (illustrative)
total_ms = sum(ms for _, ms in spans)
slow_name, slow_ms = max(spans, key=lambda s: s[1])
print("classify_review  (total {} ms)".format(total_ms))
for name, ms in spans:
    bar = "#" * round(ms / total_ms * 40)
    print("  {:<26} {:>5} ms  {:>4.0%}  {}".format(name, ms, ms / total_ms, bar))
print()
print("A flat log says only 'latency: {} ms'.".format(total_ms))
print("The trace says the {} span is {:.0%} of it ({} ms); retrieval is only {} ms.".format(slow_name, slow_ms / total_ms, slow_ms, 31))
print("Without spans you might 'optimise' retrieval and save nothing - the model call is where the time actually is.")

# Output:
# classify_review  (total 1247 ms)
#   load_prompt                    3 ms    0%  
#   embed_review                  48 ms    4%  ##
#   retrieve_similar              31 ms    2%  #
#   build_messages                 2 ms    0%  
#   anthropic.messages.create   1140 ms   91%  #####################################
#   parse_json                     1 ms    0%  
#   log_outcome                   22 ms    2%  #
#
# A flat log says only 'latency: 1247 ms'.
# The trace says the anthropic.messages.create span is 91% of it (1140 ms); retrieval is only 31 ms.
# Without spans you might 'optimise' retrieval and save nothing - the model call is where the time actually is.

- Each **span** is one sub-operation with its own timing; the tree sums to the same total a flat log would report.
- The model call is the **large majority** of the total time; retrieval is a tiny slice.
- A flat log would have shown only the total — you might have optimised retrieval and saved nothing.
- The trace localises the bottleneck in seconds; that is per-step visibility.

#### 💡 What just happened

⚡ What just happened?A trace breaks one request into per-step spans, each with its own timing and cost; a flat log gives only the total, so the trace is what tells you WHERE the time went. Tradeoff: a little more instrumentation, in exchange for five-minute debugging instead of five-hour. Next: the curated set that defines what a good answer even is.

- A bar chart of per-step span durations; the model call span dwarfs the rest
- A flat-log toggle collapses the tree into a single total

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: The Golden Set

### The Golden Set

The golden set is your single most valuable eval artifact: versioned (input, expected) pairs re-run on every change. The 2026 design has four buckets - a stratified production sample, an adversarial set, constructed edge cases, and replays of shipped failures. The HARD slice is where you learn; an easy-only pass rate hides the real weaknesses.

You cannot evaluate against “good” in the abstract, so you build a **golden set**: a versioned list of **(input, expected)** pairs that you re-run on every prompt, model, or retrieval change. It is the oldest idea in software testing — a test suite — applied to LLMs, where “expected” is sometimes a label, sometimes a rubric, sometimes “must mention X”. The 2026 design has **four buckets**: a *stratified sample of production traffic* (what real inputs look like), an *adversarial set* (jailbreaks and traps), *constructed edge cases* (the ambiguous ones humans argue about), and *replays of shipped failures* (every past bug becomes a permanent test). The single most important discipline is to include **hard cases**: a set of only easy inputs lets a mediocre prompt score a perfect pass rate while it silently fails everything ambiguous. You version the set in git and grow it every time production surprises you. The block runs a champion against a small golden set and splits the score by difficulty, keyless.

> 📝 **Analogy**
>
> A golden set is **a standardized exam with an answer key**. Every candidate sits the *same* paper, and each question has a known correct answer, so you can compare scores fairly and objectively — that is exactly what re-running the fixed (input, expected) set on every change gives you. And a good exam is not all easy questions: it deliberately includes the tricky ones that separate a strong candidate from a lucky one. An easy-only golden set is a test where everyone gets an A and you have learned nothing about who can actually do the job.

Your golden set is all easy cases and the champion scores a perfect pass rate on it. What has the eval actually told you?

**📝 `02_golden_set.py`** - *runnable*

In [ ]:
# The golden set is the eval's ground truth: versioned (input, expected) pairs re-run on every change. The 2026
# design has 4 buckets - a stratified production sample, an adversarial set, constructed edge cases, and replays of
# shipped failures. The HARD slice is where you actually learn: an easy-only pass rate hides the real weaknesses.
GOLDEN = [  # (input, expected, difficulty)
    ("pizza cold and late, refund please", "refund", "easy"),
    ("delivery boy was rude", "service", "easy"),
    ("food great but driver took forever", "delivery", "hard"),   # ambiguous: delivery vs service
    ("5 stars, loved it", "other", "easy"),
    ("mi comida estaba fria", "food_quality", "hard"),            # Spanish: "my food was cold"
    ("app crashed when paying", "service", "easy"),
    ("pasta had too much oil", "food_quality", "easy"),
    ("wrong order delivered", "delivery", "easy"),
    ("want to cancel my subscription", "refund", "easy"),
    ("how do I track my order", "other", "easy"),
]
CHAMPION = {  # a basic no-few-shot prompt: misses the two HARD cases (illustrative, deterministic)
    "pizza cold and late, refund please": "refund", "delivery boy was rude": "service",
    "food great but driver took forever": "service", "5 stars, loved it": "other",
    "mi comida estaba fria": "delivery", "app crashed when paying": "service",
    "pasta had too much oil": "food_quality", "wrong order delivered": "delivery",
    "want to cancel my subscription": "refund", "how do I track my order": "other",
}
passes = sum(1 for inp, exp, _ in GOLDEN if CHAMPION[inp] == exp)
easy = [(inp, exp) for inp, exp, d in GOLDEN if d == "easy"]
hard = [(inp, exp) for inp, exp, d in GOLDEN if d == "hard"]
easy_pass = sum(1 for inp, exp in easy if CHAMPION[inp] == exp)
hard_pass = sum(1 for inp, exp in hard if CHAMPION[inp] == exp)
print("Golden set: {} cases. Champion pass-rate: {}/{} = {:.0%}".format(len(GOLDEN), passes, len(GOLDEN), passes / len(GOLDEN)))
print("  easy: {}/{} = {:.0%}    hard: {}/{} = {:.0%}".format(easy_pass, len(easy), easy_pass / len(easy), hard_pass, len(hard), hard_pass / len(hard)))
print("  the champion aces the easy cases and FAILS every hard one - a headline pass-rate would have hidden that.")
print()
print("Re-run this exact set on every prompt/model change; version it in git; grow it by replaying each new production failure.")

# Output:
# Golden set: 10 cases. Champion pass-rate: 8/10 = 80%
#   easy: 8/8 = 100%    hard: 0/2 = 0%
#   the champion aces the easy cases and FAILS every hard one - a headline pass-rate would have hidden that.
#
# Re-run this exact set on every prompt/model change; version it in git; grow it by replaying each new production failure.

- The golden set is versioned **(input, expected)** pairs; the champion is re-run against all of them.
- The headline pass-rate looks decent, but the split by difficulty tells the real story.
- The champion **aces the easy cases and fails every hard one** — a headline number would have hidden that.
- Version the set in git; grow it by replaying each new production failure into a permanent test.

#### 💡 What just happened

⚡ What just happened?The golden set is versioned (input, expected) pairs re-run on every change, with four buckets (production sample, adversarial, edge cases, shipped-failure replays); the hard slice is where you learn. Tradeoff: hand-curation effort, in exchange for a repeatable definition of good. Next: screening outputs cheaply before you pay a judge.

- A grid of ten golden cases lights green on pass, red on fail as the champion runs
- An easy vs hard split, and the four golden-set buckets

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Code-Based Asserts

### Code-Based Asserts

Level 3 (objective) before level 4 (subjective). Cheap, instant, DETERMINISTIC code asserts - valid JSON, in-enum, length, no-PII - run FIRST and reject malformed outputs for free. Only what passes goes to the slow, paid LLM judge. No API call, no cost, no flakiness for anything a regex can catch.

Not everything needs a judge. Many failures are **objective** and a plain function catches them: the output is not valid JSON, it is not one of the allowed categories, it is too long, it leaked an email address. These are **code asserts** — cheap, instant, deterministic checks — and the discipline is to run them **first**. An output that fails a code assert is rejected for *zero cost and zero latency*; only the outputs that pass are worth sending to the slow, paid LLM judge. This is the metric pyramid in action: **level 3 (objective)** before **level 4 (subjective)**. It saves money, it removes flakiness (a regex never has a bad day), and it catches the boring format bugs that would otherwise waste a judge call. The block runs a handful of outputs through three asserts and counts how many are even worth judging, keyless.

> 🔢 **Analogy**
>
> Code asserts are **running spell-check before you pay a human editor**. You would not send a document riddled with obvious typos to a professional editor at their hourly rate — you run the free, instant spell-check first, fix the mechanical errors, and only pay the editor for the judgement calls a machine cannot make (tone, argument, clarity). Code asserts are the spell-check: the enum check, the JSON check, the no-PII check are free and instant, and they mean the expensive LLM judge only ever sees outputs that already clear the mechanical bar.

An output comes back as the string ‘REFUND!!! definitely refund’ when you expected a single lowercase category. What is the cheapest way to fail it?

**📝 `03_code_asserts.py`** - *runnable*

In [ ]:
# Level 3 (objective) BEFORE level 4 (subjective): cheap, instant, deterministic CODE ASSERTS catch format bugs for
# free - run them FIRST and only send what passes to the slow, paid LLM judge. No API, no cost, no flakiness.
CATEGORIES = {"refund", "delivery", "food_quality", "service", "other"}
import re
def code_asserts(output):
    fails = []
    if output not in CATEGORIES: fails.append("not_in_enum")
    if output != output.lower() or " " in output.strip(): fails.append("not_single_lowercase_token")
    if re.search(r"[\w.+-]+@[\w-]+\.[\w.]+", output): fails.append("contains_email_PII")
    return fails
outputs = ["refund", "REFUND!!! definitely refund", "{'category': 'refund'}",
           "contact me at u@x.com", "delivery", "banana"]
clean = 0
for o in outputs:
    fails = code_asserts(o)
    verdict = "PASS -> send to judge" if not fails else "FAIL (free) -> " + ", ".join(fails)
    if not fails: clean += 1
    print("  {:<32} {}".format(repr(o)[:32], verdict))
print()
print("{}/{} outputs pass the code asserts and are worth a judge call; the rest are rejected for $0 and 0 ms.".format(clean, len(outputs)))
print("Code asserts are deterministic and instant - they are the cheap first gate before the expensive subjective one.")

# Output:
#   'refund'                         PASS -> send to judge
#   'REFUND!!! definitely refund'    FAIL (free) -> not_in_enum, not_single_lowercase_token
#   "{'category': 'refund'}"         FAIL (free) -> not_in_enum, not_single_lowercase_token
#   'contact me at u@x.com'          FAIL (free) -> not_in_enum, not_single_lowercase_token, contains_email_PII
#   'delivery'                       PASS -> send to judge
#   'banana'                         FAIL (free) -> not_in_enum
#
# 2/6 outputs pass the code asserts and are worth a judge call; the rest are rejected for $0 and 0 ms.
# Code asserts are deterministic and instant - they are the cheap first gate before the expensive subjective one.

- Each output runs through cheap **code asserts** — in-enum, single lowercase token, no PII — that are free and instant.
- Malformed outputs are **rejected for zero cost**, with the failed assert named.
- Only the outputs that pass the asserts are worth sending to the slow, paid judge.
- Level 3 (objective, code) runs before level 4 (subjective, judge) — screen cheaply first.

#### 💡 What just happened

⚡ What just happened?Code asserts are cheap, instant, deterministic checks (enum, JSON, length, no-PII) that reject malformed outputs for free; run them first and only send what passes to the paid LLM judge. Tradeoff: none worth the name - they are strictly cheaper and more reliable than a judge for objective checks. Next: judging the subjective part that code cannot.

- Six outputs flow through three cheap asserts; each reject is stamped free
- Only the clean outputs reach the paid judge; a saved-calls counter

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: LLM-As-Judge

### LLM-As-Judge

When the output is freeform, a stronger, DIFFERENT-family LLM grades it against a rubric. The rules that make it work: multi-dimension, a SMALL discrete scale (0-2, never a free 1-10 which just clusters at 7), the reference answer in the prompt, and pass = all dimensions at least 1. It is a ten-line technique that revolutionised eval at scale.

For freeform outputs — summaries, chat replies, generated code — there is no label to match, so you use an **LLM-as-judge**: a second, usually stronger model grades the first model’s output against a **rubric**. Four rules make it trustworthy. Use **multiple dimensions** (correctness, format, calibration, tone, harm) rather than one blurry score. Use a **small discrete scale** — zero, one, two — because a free one-to-ten scale just makes the judge cluster every output at seven. Put the **reference answer in the prompt** so the judge is not guessing what “correct” means. And define **pass as all dimensions at least one**, so a single bad dimension cannot hide behind good ones. Critically, the judge should be a **stronger, different-family model** than the one under test (an Opus-class model judging a Sonnet-class one), because a model over-rewards its own family’s style. The block models the rubric logic deterministically so it runs keyless; the illustrative artifact shows the real Opus-class judge call.

> 📝 **Analogy**
>
> An LLM-as-judge is **a second examiner grading essays against a rubric**. A good exam board does not let the teacher who taught the class grade their own students’ essays with a vague “that felt like a B” — they bring in an *independent* examiner who scores against a written rubric with explicit criteria and a small band scale, so the marks are comparable and defensible. The LLM judge is that independent examiner: a different, stronger model, a multi-dimension rubric, a small discrete scale, and the model answer in front of them — not a gut feeling on a one-to-ten line.

You grade freeform answers by asking a judge for one overall score from 1 to 10. What goes wrong?

**📝 `04_llm_judge.py`** - *runnable*

In [ ]:
# Level 4 (subjective): when the output is freeform, an LLM-AS-JUDGE grades it against a RUBRIC. Rules that make it
# work: multi-dimension, a SMALL discrete scale (0-2, never free 1-10), ground truth in the prompt, pass = all dims >= 1.
def judge_rubric(expected, output):   # models the judge's rubric logic deterministically (real judge = a stronger model)
    correctness = 2 if output == expected else 0
    fmt = 2 if (output == output.lower() and " " not in output.strip()) else 1
    calibration = 2                    # no hedging in a one-word label
    passed = correctness >= 1 and fmt >= 1 and calibration >= 1
    return {"correctness": correctness, "format": fmt, "calibration": calibration, "pass": passed}
cases = [("refund", "refund"), ("delivery", "service"), ("other", "other"),
         ("food_quality", "Food quality."), ("refund", "refund")]
passes = 0
for expected, output in cases:
    v = judge_rubric(expected, output)
    if v["pass"]: passes += 1
    print("  expected={:<12} output={:<16} -> correctness={} format={} calib={}  {}".format(
          expected, repr(output)[:16], v["correctness"], v["format"], v["calibration"], "PASS" if v["pass"] else "FAIL"))
print()
print("Judge pass-rate on these {} cases: {}/{}. A discrete rubric gives a structured verdict, not a vague '7/10'.".format(len(cases), passes, len(cases)))
print("Use a stronger, DIFFERENT-family model as judge, and put the expected/reference answer in the prompt. Then calibrate it.")

# Output:
#   expected=refund       output='refund'         -> correctness=2 format=2 calib=2  PASS
#   expected=delivery     output='service'        -> correctness=0 format=2 calib=2  FAIL
#   expected=other        output='other'          -> correctness=2 format=2 calib=2  PASS
#   expected=food_quality output='Food quality.'  -> correctness=0 format=1 calib=2  FAIL
#   expected=refund       output='refund'         -> correctness=2 format=2 calib=2  PASS
#
# Judge pass-rate on these 5 cases: 3/5. A discrete rubric gives a structured verdict, not a vague '7/10'.
# Use a stronger, DIFFERENT-family model as judge, and put the expected/reference answer in the prompt. Then calibrate it.

**📝 `llm_judge_real.py`** - *illustrative*

In [ ]:
# PRODUCTION LLM-AS-JUDGE - an illustrative minimal example (needs the Anthropic SDK + a key; not run here).
import os, json
from anthropic import Anthropic
client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"], timeout=30)

JUDGE_PROMPT = """You are an evaluator grading a customer-support classifier.
Task: classify the review into one of [refund, delivery, food_quality, service, other].
Review: {review}
Expected category (ground truth from a senior agent): {expected}
Assistant output: {output}
Grade each dimension 0-2 (0 bad, 1 ok, 2 good):
1. correctness - does the output match the expected category?
2. format - is the output ONLY the lowercase category name?
3. calibration - if it hedged, was the uncertainty warranted?
Reply with JSON ONLY:
{{"correctness": 0|1|2, "format": 0|1|2, "calibration": 0|1|2, "rationale": "1 sentence", "pass": true if all >= 1 else false}}"""

def judge(review, expected, output):
    rendered = JUDGE_PROMPT.format(review=review, expected=expected, output=output)
    r = client.messages.create(          # a STRONGER, DIFFERENT-family model than the one under test
        model="claude-opus-4-8", max_tokens=300,
        messages=[{"role": "user", "content": rendered}])
    return json.loads(r.content[0].text)  # then calibrate: check kappa vs human labels before you trust it
# Output: (illustrative - needs the Anthropic SDK + ANTHROPIC_API_KEY; a real judge call, not run here.)

- The judge scores each output on a **discrete 0-2 rubric** (correctness, format, calibration); pass = all dimensions at least one.
- A structured multi-dimension verdict beats a vague “seven out of ten” — you can see *which* dimension failed.
- The **illustrative artifact** is the real judge: a stronger, different-family model with the reference answer in the prompt.
- The verdict is only as good as the judge — which is why the next step calibrates it against humans.

#### 💡 What just happened

⚡ What just happened?An LLM-as-judge grades freeform output against a small discrete multi-dimension rubric, with the reference answer in the prompt and pass = all dimensions at least one; use a stronger, different-family model. Tradeoff: a judge is itself a fallible model, so you must calibrate it. Next: how you know whether to trust the judge at all.

- One output scored on a discrete 0-2 rubric (correctness, format, calibration)
- A free 1-10 slider that clusters at 7, beside the crisp rubric

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Judge Calibration

### Judge Calibration

You do not trust a judge until it agrees with HUMANS. Raw agreement flatters, so measure Cohen’s kappa, which corrects for chance agreement (bar ~0.6; well-calibrated judges reach 80-90 percent within-one-point). And a side-by-side judge favours the FIRST option (position bias ~10-15 points) and its own family (self-preference ~5-7 percent) - so average both orderings and use a different family.

The judge is itself a model, so the question “is the judge right?” is as real as “is the model right?”. You answer it by **calibration**: hand-label a sample of cases and measure how well the judge agrees with the humans. The trap is trusting **raw agreement**, which flatters — two labellers agree by chance a large fraction of the time, so eighty-three percent agreement can be worse than it sounds. The fix is **Cohen’s kappa**, which subtracts the agreement you would expect by chance; a common production bar is around **0.6**, and a well-calibrated judge reaches eighty to ninety percent within-one-point agreement on a clear rubric. Below the bar, the task is too subjective for labels and you need better rubrics. Two documented biases also need controlling: in a side-by-side comparison the judge favours the **first option** (position bias, worth ten to fifteen points), and it over-rewards its **own family** (self-preference, about five to seven percent). Mitigate by running both orderings and averaging, and by using a different family. The block computes agreement, kappa, and the position bias, keyless.

> 🏅️ **Analogy**
>
> Calibration is **checking a new judge’s scores against the head judge before the finals**. A gymnastics competition does not just trust a new official because their scores “seem about right” — they seat the newcomer next to the experienced head judge through the qualifiers and check how closely their marks track, correcting for the fact that two people scoring a ten-point scale will land on the same number now and then by luck. Only once the new judge’s agreement is genuinely high — beyond chance — do they score the finals alone. Cohen’s kappa is exactly that “beyond chance” correction, and the position-bias check is making sure the judge is not just always favouring whoever performs first.

Your LLM judge agrees with human labels eighty-three percent of the time. Is that good enough to trust?

**📝 `05_judge_calibration.py`** - *runnable*

In [ ]:
# You do NOT trust a judge until it agrees with HUMANS. Raw agreement flatters; Cohen's kappa corrects for chance
# agreement (bar ~0.6). And a side-by-side judge favours the FIRST option (position bias) - so average both orderings.
human = [1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1]   # 12 calibration cases, human pass/fail (the ground truth)
judge = [1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1]   # the judge disagrees on 2 of them
n = len(human)
agree = sum(1 for h, j in zip(human, judge) if h == j)
po = agree / n
p_h1, p_j1 = sum(human) / n, sum(judge) / n
pe = p_h1 * p_j1 + (1 - p_h1) * (1 - p_j1)      # expected agreement by chance
kappa = (po - pe) / (1 - pe)
print("Calibration ({} cases): raw agreement {}/{} = {:.0%}   Cohen's kappa = {:.2f}".format(n, agree, n, po, kappa))
verdict = "TRUSTWORTHY (kappa >= 0.6)" if kappa >= 0.6 else "NOT YET (kappa < 0.6) - fix the rubric before you trust the scores"
print("  -> {}".format(verdict))
print("  {:.0%} agreement LOOKS fine, but kappa {:.2f} shows it is only borderline once you subtract chance agreement.".format(po, kappa))
print()
first_slot_wins, pairs = 13, 20                 # same 2 outputs, judged A-then-B and B-then-A (illustrative)
print("Position bias: the judge picks the FIRST slot {}/{} = {:.0%} of the time (>55% = biased).".format(first_slot_wins, pairs, first_slot_wins / pairs))
print("Mitigate: run BOTH orderings and average; use a different model family; self-preference bias is a ~5-7% own-family boost.")

# Output:
# Calibration (12 cases): raw agreement 10/12 = 83%   Cohen's kappa = 0.56
#   -> NOT YET (kappa < 0.6) - fix the rubric before you trust the scores
#   83% agreement LOOKS fine, but kappa 0.56 shows it is only borderline once you subtract chance agreement.
#
# Position bias: the judge picks the FIRST slot 13/20 = 65% of the time (>55% = biased).
# Mitigate: run BOTH orderings and average; use a different model family; self-preference bias is a ~5-7% own-family boost.

- **Raw agreement** looks reassuring, but two labellers agree by chance a large fraction of the time.
- **Cohen’s kappa** subtracts that chance agreement — here it lands below the usual bar, so the judge is only borderline.
- So a healthy-looking agreement number can still mean “fix the rubric before you trust the scores”.
- **Position bias** shows the judge favouring the first slot — run both orderings and average, and use a different family.

#### 💡 What just happened

⚡ What just happened?You calibrate a judge against humans: raw agreement flatters, so measure Cohen’s kappa (bar ~0.6) and control the biases - position (average both orderings) and self-preference (different family). Tradeoff: hand-labelling a calibration set, in exchange for numbers you can actually trust. Next: running the calibrated judge automatically on every change.

- A judge-vs-human grid with a big agreement number beside a Cohen kappa gauge
- A position-bias meter above the 55 percent line; average both orderings

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: The CI Eval Gate

### The CI Eval Gate

Your golden set is only useful if it runs automatically before merges. Every prompt, model, or retrieval PR triggers a golden-set eval, and the gate BLOCKS the merge unless three checks pass: an absolute pass-rate FLOOR, a max REGRESSION versus main, and a COST ceiling. A green eval suite is the ship gate - eval-driven development. The thresholds are grounded in the significance math of 14.2.

An eval you run by hand is an eval you forget to run. The automation that makes you a level-3 team is the **CI eval gate** (the `ci-eval-gate` stage of your pipeline): every pull request that touches a prompt, a model version, or a retrieval config triggers a golden-set eval, posts the delta as a PR comment, and **exits non-zero to block the merge** if the change is bad. This is **eval-driven development** — a green eval suite is the ship gate, the same way a green test suite is. The gate checks **three** things, because a change can be bad in three different ways: an **absolute floor** (the candidate must clear the product quality bar on its own merits), a **relative regression limit** (it must not be meaningfully worse than what you already ship), and a **cost ceiling** (it must not blow up the per-request bill). The thresholds are not arbitrary: the roughly two-point regression limit is about the smallest lift a two-hundred-case golden set can even *detect* — the significance math from Lesson 14.2. The block runs three PR scenarios through the gate, keyless.

> 👮 **Analogy**
>
> The CI eval gate is **a bouncer with a three-item checklist**. A good bouncer does not just check one thing — they check the ID is valid (are you good enough on your own? the absolute floor), that you are not causing trouble compared to the regulars already inside (are you worse than what we have? the regression limit), and that you are not going to run up a tab you cannot cover (the cost ceiling). Fail *any one* and you do not get in. The eval gate is that door: three independent checks, and a single failure blocks the merge, no matter how good the other two look.

A candidate prompt scores well on its own but costs fifty percent more per request than main. What should the three-part gate do?

**📝 `06_ci_eval_gate.py`** - *runnable*

In [ ]:
# The CI EVAL GATE runs the golden set on every PR and BLOCKS the merge unless three checks pass: an absolute pass-rate
# FLOOR, a max REGRESSION vs main, and a COST ceiling. It exits non-zero to fail the pull request. (Significance math: 14.2.)
def eval_gate(base_pass, pr_pass, base_cost, pr_cost, floor=0.92, max_drop=0.02, max_cost_incr=0.30):
    reasons = []
    if pr_pass < floor: reasons.append("below floor {:.0%} (got {:.0%})".format(floor, pr_pass))
    if base_pass - pr_pass > max_drop: reasons.append("regressed {:.0f}pp vs main (max {:.0f}pp)".format((base_pass - pr_pass) * 100, max_drop * 100))
    if (pr_cost - base_cost) / base_cost > max_cost_incr: reasons.append("cost +{:.0%} vs main (max +{:.0%})".format((pr_cost - base_cost) / base_cost, max_cost_incr))
    return ("PASS - merge allowed", reasons) if not reasons else ("BLOCK - merge stopped", reasons)
scenarios = [("clean improvement", 0.92, 0.96, 0.0016, 0.0016),
             ("quality regression", 0.92, 0.89, 0.0016, 0.0016),
             ("too expensive",      0.92, 0.95, 0.0016, 0.0024)]
for name, bp, pp, bc, pc in scenarios:
    decision, reasons = eval_gate(bp, pp, bc, pc)
    print("  {:<20} base {:.0%} -> pr {:.0%}, cost ${:.4f} -> ${:.4f}  ::  {}".format(name, bp, pp, bc, pc, decision))
    for r in reasons:
        print("        - {}".format(r))
print()
print("Every prompt/model/retrieval PR triggers the gate; a green eval suite is the ship gate (eval-driven development).")
print("Thresholds are not arbitrary: ~2pp is about the smallest lift a 200-case golden set can even detect (Lesson 14.2).")

# Output:
#   clean improvement    base 92% -> pr 96%, cost $0.0016 -> $0.0016  ::  PASS - merge allowed
#   quality regression   base 92% -> pr 89%, cost $0.0016 -> $0.0016  ::  BLOCK - merge stopped
#         - below floor 92% (got 89%)
#         - regressed 3pp vs main (max 2pp)
#   too expensive        base 92% -> pr 95%, cost $0.0016 -> $0.0024  ::  BLOCK - merge stopped
#         - cost +50% vs main (max +30%)
#
# Every prompt/model/retrieval PR triggers the gate; a green eval suite is the ship gate (eval-driven development).
# Thresholds are not arbitrary: ~2pp is about the smallest lift a 200-case golden set can even detect (Lesson 14.2).

- Every prompt/model/retrieval PR triggers the golden-set eval; the gate returns PASS or BLOCK.
- The gate checks **three** things: an absolute floor, a max regression versus main, and a cost ceiling.
- A clean improvement passes; a quality regression **and** a cost blow-up each block the merge, with the reason named.
- The thresholds come from the significance math of 14.2 — the regression limit is about the smallest lift the set can detect.

#### 💡 What just happened

⚡ What just happened?The CI eval gate runs the golden set on every change and BLOCKS the merge unless three checks pass - an absolute floor, a max regression versus main, and a cost ceiling; a green suite is the ship gate. Tradeoff: some eval cost per PR, in exchange for never shipping a silent regression. Next: catching regressions that appear with no change at all.

- Three PRs pass through a three-check gate (floor, regression, cost)
- The failing check lights up and the merge is blocked

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Drift Detection

### Drift Detection

The scariest failure mode: nothing in your code changed, but quality dropped. In LLMOps, drift includes a SILENT provider model update (they ship faster than you release), a stale RAG index, or aging prompts. Run the golden set NIGHTLY against the production champion and alert on a >2pp week-over-week drop. Four signals: pass-rate, refusal rate, format-compliance, cost per request.

The CI gate protects you from bad *changes*. Drift is the failure with **no change at all**: your code is untouched, but quality falls. In classical machine learning drift is usually input-distribution shift; in LLMOps it also includes a **silent provider model update** (the provider ships an internal tweak to the model you pinned, and behaviour subtly shifts — and providers ship far more often than you release), a **stale RAG index**, or a **prompt aging** as user phrasings change. The defence is **continuous evaluation**: run your golden set **nightly** against the production champion, log the pass-rate over time, and alert on a drop of more than about two points week-over-week. Four signals are worth alerting on: golden-set pass-rate, refusal rate, format-compliance rate, and cost per request. The insidious part is that this drift is invisible to your ordinary monitoring — the error rate stays green while the answers quietly get worse (the fourth failure mode from Lesson 12.8). The block runs a week of nightly evals and fires the alert, keyless.

> ⚖️ **Analogy**
>
> Drift detection is **a nightly weigh-in that catches slow weight gain nobody noticed**. If you only step on the scale when you feel unwell, you miss the pound-a-week creep that adds up to a serious problem over a season — each day feels the same as the last. Stepping on the scale every night and watching the trend catches the slow drift long before it becomes a crisis, even though nothing dramatic happened on any single day. Running your golden set every night is that weigh-in: no single day looks alarming, but the trend line reveals the quiet six-point slide, and the two-point alert is the line on the chart that says “go look now”.

Your golden-set pass-rate fell six points over a week and nobody changed the prompt or the code. Most likely cause?

**📝 `07_drift_detection.py`** - *runnable*

In [ ]:
# Drift: nothing in YOUR code changed, but quality falls. Run the golden set NIGHTLY against the production champion and
# alert on a >2pp week-over-week drop - the usual culprit is a SILENT provider model update (they ship faster than you release).
nightly = [0.95, 0.95, 0.94, 0.95, 0.93, 0.90, 0.89, 0.89]   # champion pass-rate per night, no prompt change (illustrative)
week_start = nightly[0]
ALERT_DROP = 0.02
print("day  pass-rate  vs week-start  status")
for day, rate in enumerate(nightly):
    drop = week_start - rate
    status = "ALERT (drop > {:.0f}pp)".format(ALERT_DROP * 100) if drop > ALERT_DROP else "ok"
    print("  {:>2}    {:.0%}        {:+.0f}pp         {}".format(day, rate, -drop * 100, status))
print()
print("The champion fell from {:.0%} to {:.0%} ({:+.0f}pp) over the week with ZERO code changes -> suspect a provider model update.".format(
      week_start, nightly[-1], -(week_start - nightly[-1]) * 100))
print("Four drift signals to alert on: golden-set pass-rate, refusal rate, format-compliance rate, cost per request.")
print("Nightly eval costs a few dollars and catches quality regressing while error rate stays green (the 12.8 4th failure mode).")

# Output:
# day  pass-rate  vs week-start  status
#    0    95%        -0pp         ok
#    1    95%        -0pp         ok
#    2    94%        -1pp         ok
#    3    95%        -0pp         ok
#    4    93%        -2pp         ok
#    5    90%        -5pp         ALERT (drop > 2pp)
#    6    89%        -6pp         ALERT (drop > 2pp)
#    7    89%        -6pp         ALERT (drop > 2pp)
#
# The champion fell from 95% to 89% (-6pp) over the week with ZERO code changes -> suspect a provider model update.
# Four drift signals to alert on: golden-set pass-rate, refusal rate, format-compliance rate, cost per request.
# Nightly eval costs a few dollars and catches quality regressing while error rate stays green (the 12.8 4th failure mode).

- The nightly golden-set pass-rate is logged over time; the champion is unchanged.
- It **slides down over the week** and crosses the two-point alert threshold, firing a standing alert.
- The pass-rate fell several points with **zero code changes** — the classic signature of a silent provider model update.
- Four signals to watch: pass-rate, refusal rate, format-compliance, and cost per request — drift can hide in any of them.

#### 💡 What just happened

⚡ What just happened?Drift is quality falling with nothing in your code changing - usually a silent provider model update; run the golden set nightly, alert on a two-point week-over-week drop, and watch four signals. Tradeoff: a few dollars a night, in exchange for catching regression the error rate never shows. That is the whole loop: trace it, curate it, screen it, judge it, calibrate it, gate it, and watch it.

- A nightly pass-rate line slides down past a two-point threshold line
- The threshold trips a standing alert; four drift signals

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> “Did the model do a good job?” becomes three answerable questions. **What did it do?** is the **trace** — per-step spans that localise time and cost (Step 1). **Did it do right on a curated set?** is the **golden set** (Step 2), screened cheaply by **code asserts** (Step 3) and graded on the subjective parts by an **LLM-as-judge** (Step 4) that you only trust once its **kappa** clears the bar (Step 5). **Is it still right today?** is the **CI eval gate** on every change (Step 6) plus **drift detection** on live traffic (Step 7). Ask two questions of any LLM feature: can you see what it actually did, and do you have a calibrated, automated way to know if it is getting worse?

**📦 **The metric pyramid****

| Level | Metric | How hard | Answers |
|---|---|---|---|
| 1 | Operational— latency, error rate, throughput | easy (from tracing) | is it up and fast? |
| 2 | Economic— cost per request / user / feature | easy (from tracing) | will we run out of budget? |
| 3 | Objective— valid JSON, in-enum, length, no-PII | medium (code asserts + golden set) | is the output usable? |
| 4 | Subjective— helpfulness, factuality, tone, harm | hard (calibrated LLM-as-judge) | is the output good? |

> 📦 **Info**
>
> ➡️ Where this goes nextYou can now measure quality — but this lesson assumed you already knew *what* to measure (which categories, which rubric dimensions, which failures matter). Deciding that, by looking at real traces and building a failure taxonomy through open and axial coding, is **error analysis** — the fastest-growing LLMOps skill of 2026, and it comes next, in Lesson 14.4, where error analysis decides what your judges and your CI eval gate should even measure. Serving the evaluated winner cheaply, with caching and routing, comes in Lesson 14.5. The primary references are the [DeepEval](https://github.com/confident-ai/deepeval) and [RAGAS](https://github.com/explodinggradients/ragas) eval frameworks, the [Langfuse scores docs](https://langfuse.com/docs/scores/overview), and the [OpenTelemetry GenAI conventions](https://github.com/open-telemetry/semantic-conventions/tree/main/docs/gen-ai).

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Observability& Evaluation**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-14_3.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-14.3.html` - regenerate this notebook whenever the source HTML is updated.*
